In [ ]:
import os
import zipfile

from google.colab import drive
drive.mount('/content/drive')

caminho_zip = "/content/drive/MyDrive/PlantVillage/archive.zip"
pasta_temp_pv = "/content/PlantVillage_Bruto"

print("Extraindo PlantVillage para a memória rápida do Colab...")
with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_temp_pv)
print("Extração concluída!")

In [ ]:
import glob
import random
import shutil
from tqdm import tqdm

# Busca recursiva pelas duas pastas exatas que importam para o escopo
try:
    pasta_saudavel = glob.glob(f"{pasta_temp_pv}/**/Tomato___healthy", recursive=True)[0]
    pasta_doente = glob.glob(f"{pasta_temp_pv}/**/Tomato___Early_blight", recursive=True)[0]
except IndexError:
    print("Erro: Não foi possível encontrar as pastas exatas de Pinta-Preta e Saudável. Verifique a estrutura extraída.")

OUTPUT_DOENCA = "/content/dataset_doenca_otimizado"

# Cria a estrutura de pastas final (treino, validacao, teste)
splits = ['treino', 'validacao', 'teste']
classes = ['saudavel', 'pinta_preta']
for s in splits:
    for c in classes:
        os.makedirs(os.path.join(OUTPUT_DOENCA, s, c), exist_ok=True)

def processar_classe(caminho_origem, nome_classe_destino):
    # Pega todas as imagens (jpg, JPG, png)
    imagens = []
    for ext in ['*.jpg', '*.JPG', '*.png']:
        imagens.extend(glob.glob(os.path.join(caminho_origem, ext)))
    
    # Embaralha as imagens com seed fixa para reprodutibilidade
    random.seed(42)
    random.shuffle(imagens)
    
    total = len(imagens)
    idx_treino = int(total * 0.70)
    idx_val = int(total * 0.85)
    
    splits_imgs = {
        'treino': imagens[:idx_treino],
        'validacao': imagens[idx_treino:idx_val],
        'teste': imagens[idx_val:]
    }
    
    for split_name, imgs in splits_imgs.items():
        destino = os.path.join(OUTPUT_DOENCA, split_name, nome_classe_destino)
        for img in tqdm(imgs, desc=f"Copiando {nome_classe_destino} -> {split_name}"):
            shutil.copy(img, destino)

print("Iniciando split das folhas saudáveis...")
processar_classe(pasta_saudavel, 'saudavel')

print("\nIniciando split das folhas com Pinta-Preta...")
processar_classe(pasta_doente, 'pinta_preta')

# Compacta e envia para a mesma pasta PlantVillage no seu Drive
destino_zip = "/content/drive/MyDrive/PlantVillage/dataset_doenca_otimizado"
print(f"\nCompactando dataset final de doenças para o Drive ({destino_zip}.zip)...")
shutil.make_archive(destino_zip, 'zip', OUTPUT_DOENCA)
print("Processo concluído com sucesso!")